# PostProcessing pyspellchecker (Billex & Morphology)

Companion to `PostProcessing pyspellchecker.ipynb`, which only handles the
Parallel Corpus. This notebook applies the same `pyspellchecker` pipeline,
with the same Indonesian dictionary, to the **other two NLP resources**:

- **Bilingual Lexicon** (`9. Bilingual Lexicon - Fixed/*_Billex.csv`)
- **Morphology** (`10. Morphology - Fixed/*_Morphology.csv`)

## Which column gets corrected?

`LookupIsFromIndonesia.csv` says whether each dictionary is Indonesian→Regional
(`is_from_indonesia=1`) or Regional to Indonesian (`is_from_indonesia=0`). Only
the **Indonesian** side is run through the spellchecker; the regional side is
left untouched because the Indonesian lexicon would falsely "correct" perfectly
valid regional words.

| Resource    | dir=1 (Ind→Reg) Indonesian column(s) | dir=0 (Reg→Ind) Indonesian column(s) |
|-------------|--------------------------------------|--------------------------------------|
| Billex      | `kata_asal`                          | `kata_tujuan`                        |
| Morphology  | `kata` and `form`                    | *(skipped — see below)*              |

**Why Morphology dir=0 is skipped.** When the source dictionary is
Regional to Indonesian, the morphology pipeline stores the **regional** derived
form in the `form` column and the **regional** headword in `kata`. The
Indonesian glosses live in `makna` upstream and aren't carried into the
Morphology resource. Running the spellchecker on regional words would corrupt
them. These rows are passed through unchanged.

## Outputs

- `../Ekstraksi/13. Bilingual Lexicon - Spelling Checker/<id>_Billex.csv`
- `../Ekstraksi/13. Bilingual Lexicon - Spelling Checker/<id>_Billex_audit.csv`
- `../Ekstraksi/14. Morphology - Spelling Checker/<id>_Morphology.csv`
- `../Ekstraksi/14. Morphology - Spelling Checker/<id>_Morphology_audit.csv`
- `_billex_summary.csv` / `_morph_summary.csv` — per-dict correction counts


## 1. Imports & paths

In [28]:
import os
import re
import sys
from pathlib import Path
from typing import Optional

import pandas as pd
from spellchecker import SpellChecker
import concurrent.futures

# Reuse the same helpers used by the morphology / billex / parcor fix scripts
sys.path.insert(0, str(Path('.').resolve()))
from _common import (
    parse_dict_id,
    load_direction_lookup,
    direction_for,
    roles_for_billex,
    roles_for_morphology,
)

# Input directories — same as PostProcessing pyspellchecker.ipynb uses
BILLEX_SRC = Path('../Ekstraksi/9. (New) Bilingual Lexicon - Fixed')
MORPH_SRC  = Path('../Ekstraksi/10. (New) Morphology - Fixed')

# Output directories — sit alongside `12. Parallel Corpus - Spelling Checker`
BILLEX_DST = Path('../Ekstraksi/13. (New) Bilingual Lexicon - Spellcheck Detection/dict_strategy_C')
MORPH_DST  = Path('../Ekstraksi/14. (New) Morphology - Spellcheck Detection/dict_strategy_C')

BILLEX_DST.mkdir(parents=True, exist_ok=True)
MORPH_DST.mkdir(parents=True, exist_ok=True)

assert BILLEX_SRC.exists(), f'Billex source dir not found: {BILLEX_SRC.resolve()}'
assert MORPH_SRC.exists(),  f'Morph source dir not found:  {MORPH_SRC.resolve()}'

print(f'Reading Billex from: {BILLEX_SRC.resolve()}')
print(f'Reading Morph  from: {MORPH_SRC.resolve()}')
print(f'Writing Billex to:   {BILLEX_DST.resolve()}')
print(f'Writing Morph  to:   {MORPH_DST.resolve()}')

Reading Billex from: C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\Ekstraksi\9. (New) Bilingual Lexicon - Fixed
Reading Morph  from: C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\Ekstraksi\10. (New) Morphology - Fixed
Writing Billex to:   C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\Ekstraksi\13. (New) Bilingual Lexicon - Spellcheck Detection\dict_strategy_C
Writing Morph  to:   C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\Ekstraksi\14. (New) Morphology - Spellcheck Detection\dict_strategy_C


## 2. Load the Indonesian spellchecker dictionary

Identical to the Parcor notebook: same dictionary, same `pyspellchecker`
configuration. Swap `DICT_PATH` between the three strategies (A baseline,
B all-tier-eligible, C ≥3-dict-frequency) the same way you would for Parcor.

Loading the dictionary once at module level keeps the spellchecker stateless
across threads.

In [29]:
lookup_df = pd.read_csv('../Ekstraksi/12. Parallel Corpus - Spelling Checker/LookupIsFromIndonesia.csv')

spell = SpellChecker(language=None)
# DICT_PATH = '../spellCheckDicts/vocab_strategies_v3_1/dict_strategy_original.json'
# DICT_PATH = '../spellCheckDicts/vocab_strategies_v3_1/dict_strategy_B.json'
DICT_PATH = '../spellCheckDicts/vocab_strategies_v3_1/dict_strategy_C.json'

if os.path.exists(DICT_PATH):
    spell.word_frequency.load_dictionary(DICT_PATH)
    print(f'Loaded JSON dictionary from {DICT_PATH}')
else:
    raise FileNotFoundError(
        f"Custom dictionary not found at '{DICT_PATH}'. "
        'Check that the working directory is correct before running.'
    )

# Direction lookup keyed by dict_id (reused for both resources)
DIRECTION = load_direction_lookup()
print(f'Loaded direction for {len(DIRECTION)} dictionaries')

Loaded JSON dictionary from ../spellCheckDicts/vocab_strategies_v3_1/dict_strategy_C.json
Loaded direction for 82 dictionaries


## 3. Core spellchecker helpers

`fix_word` and `fix_sentence` are lifted directly from
`PostProcessing pyspellchecker.ipynb` so the correction behaviour is
**identical** to what was applied to Parcor, same case-preservation, same
`spell.unknown` gate, same `spell.correction` call.

`fix_word` is exposed separately because Billex/Morph cells are typically
single tokens, not sentences. `fix_sentence` is kept for cells that contain
multi-word headwords (Billex sometimes has these, e.g. `"abdi dalem"`).

In [30]:
def fix_word(word: str) -> str:
    """Correct a single token, preserving original case."""
    if not word or not word.isalpha():
        return word
    lower_word = word.lower()
    if lower_word in spell.unknown([lower_word]):
        correction = spell.correction(lower_word)
        if correction and correction != lower_word:
            if word.istitle():
                return correction.capitalize()
            elif word.isupper():
                return correction.upper()
            else:
                return correction
    return word


def fix_sentence(sentence) -> str:
    """Run the spellchecker over every alphabetic run in a string."""
    def replace(match):
        word = match.group(0)
        lower_word = word.lower()
        if lower_word in spell.unknown([lower_word]):
            correction = spell.correction(lower_word)
            if correction and correction != lower_word:
                if word.istitle():
                    return correction.capitalize()
                elif word.isupper():
                    return correction.upper()
                else:
                    return correction
        return word
    return re.sub(r'[A-Za-z]+', replace, str(sentence))


def fix_cell(value) -> str:
    """
    Spellcheck a Billex / Morphology cell. Single-word cells go through
    `fix_word` so we never insert spurious whitespace; multi-word cells fall
    back to `fix_sentence` so each token is treated independently.
    """
    if pd.isna(value):
        return value
    s = str(value)
    if not s.strip():
        return s
    # Single alphabetic token? use fix_word for cleanest case handling.
    if re.fullmatch(r"[A-Za-z]+", s.strip()):
        return fix_word(s.strip()) if s == s.strip() else s.replace(s.strip(), fix_word(s.strip()))
    return fix_sentence(s)

## 4. Direction lookup wrapper

Mirrors `get_is_from_indonesia` from the Parcor notebook so the file-name to
direction resolution is consistent across all three resources. Returns
`(success, is_from_indonesia)` to match the original signature.

In [31]:
def get_is_from_indonesia(input_path) -> tuple[bool, int]:
    filename = os.path.basename(str(input_path))
    file_id = parse_dict_id(filename)
    if file_id is None:
        return False, -1
    if file_id in DIRECTION:
        return True, DIRECTION[file_id]
    return False, -1


# Sanity check on a few known dicts
for fname in ['4_Billex.csv', '18_Morphology.csv', '24_Billex.csv']:
    print(f'  {fname:<22} -> {get_is_from_indonesia(fname)}')

  4_Billex.csv           -> (True, 1)
  18_Morphology.csv      -> (True, 0)
  24_Billex.csv          -> (True, 0)


## 5. Billex processor

For each `<id>_Billex.csv`:

1. Look up direction: know which column is Indonesian.
2. Spellcheck every cell in the Indonesian column.
3. Leave the regional column and `makna` untouched.
4. Write the pipeline-compatible CSV and a tracking audit CSV.

The audit file keeps the original column under `<col>_original` and adds a
boolean `<col>_corrected` flag so downstream evaluation can quantify how much
the spellchecker actually changed.

In [32]:
def process_billex(file_name: str) -> str:
    try:
        input_file  = BILLEX_SRC / file_name
        output_file = BILLEX_DST / file_name
        audit_file  = BILLEX_DST / file_name.replace('.csv', '_audit.csv')

        # Skip audit files coming in from the upstream stage — we'll write our own.
        if file_name.endswith('_audit.csv'):
            return f'Skip: {file_name} (audit file from upstream)'

        success, is_from_indonesia = get_is_from_indonesia(input_file)
        if not success:
            return f'Skip: No matching ID for {file_name}'

        df = pd.read_csv(input_file)
        if 'kata_asal' not in df.columns or 'kata_tujuan' not in df.columns:
            return f'Skip: {file_name} missing kata_asal/kata_tujuan'

        ind_col, _ = roles_for_billex(is_from_indonesia)

        original = df[ind_col].astype(str).where(df[ind_col].notna(), '')
        corrected = original.apply(fix_cell)

        # Pipeline-compatible output: only the standard columns
        out_df = df.copy()
        out_df[ind_col] = corrected
        # Preserve same column order as input
        out_df.to_csv(output_file, index=False)

        # Audit CSV: original + corrected + flag
        audit_df = df.copy()
        audit_df[f'{ind_col}_original']  = original
        audit_df[ind_col]                = corrected
        audit_df[f'{ind_col}_corrected'] = original != corrected
        audit_df['direction']            = is_from_indonesia
        audit_df.to_csv(audit_file, index=False)

        n_changed = int((original != corrected).sum())
        return f'Processed: {file_name} ({len(out_df)} rows, {n_changed} corrections in {ind_col})'

    except Exception as e:
        return f'Error in {file_name}: {e}'


billex_files = [
    f for f in os.listdir(BILLEX_SRC)
    if f.endswith('.csv') and not f.endswith('_audit.csv')
    and not f.startswith('_')
]
print(f'Found {len(billex_files)} Billex files to process')

billex_results = []
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(process_billex, f) for f in billex_files]
    for count, future in enumerate(concurrent.futures.as_completed(futures), 1):
        msg = future.result()
        billex_results.append(msg)
        print(f'[{count}/{len(billex_files)}] {msg}')

Found 82 Billex files to process
[1/82] Processed: 13_Billex.csv (412 rows, 130 corrections in kata_asal)
[2/82] Processed: 11_Billex.csv (4896 rows, 525 corrections in kata_tujuan)
[3/82] Processed: 17_Billex.csv (1161 rows, 157 corrections in kata_tujuan)
[4/82] Processed: 16_Billex.csv (2044 rows, 948 corrections in kata_asal)
[5/82] Processed: 15_Billex.csv (2800 rows, 1190 corrections in kata_asal)
[6/82] Processed: 20_Billex.csv (1761 rows, 35 corrections in kata_tujuan)
[7/82] Processed: 18_Billex.csv (9116 rows, 1359 corrections in kata_tujuan)
[8/82] Processed: 22_Billex.csv (2224 rows, 114 corrections in kata_tujuan)
[9/82] Processed: 1_Billex.csv (2343 rows, 410 corrections in kata_tujuan)
[10/82] Processed: 0_Billex.csv (2328 rows, 1265 corrections in kata_asal)
[11/82] Processed: 23_Billex.csv (2001 rows, 345 corrections in kata_tujuan)
[12/82] Processed: 10_Billex.csv (4356 rows, 2572 corrections in kata_asal)
[13/82] Processed: 21_Billex.csv (1686 rows, 702 corrections i

## 6. Morphology processor

For each `<id>_Morphology.csv`:

- **dir=1 (Indonesian to Regional)**: both `kata` (headword) and `form` (derived
  form) are Indonesian, spellcheck both.
- **dir=0 (Regional to Indonesian)**: `kata` and `form` are in the regional
  language — passthrough, do not corrupt with the Indonesian lexicon.

The `tag` column is never touched (it's a POS tag, not natural-language
content). The audit file follows the same convention as Billex: original
copies are preserved and a `_corrected` flag is added per column.

In [33]:
def process_morph(file_name: str) -> str:
    try:
        input_file  = MORPH_SRC / file_name
        output_file = MORPH_DST / file_name
        audit_file  = MORPH_DST / file_name.replace('.csv', '_audit.csv')

        if file_name.endswith('_audit.csv'):
            return f'Skip: {file_name} (audit file from upstream)'

        success, is_from_indonesia = get_is_from_indonesia(input_file)
        if not success:
            return f'Skip: No matching ID for {file_name}'

        df = pd.read_csv(input_file)
        if 'kata' not in df.columns or 'form' not in df.columns:
            return f'Skip: {file_name} missing kata/form columns'

        # roles_for_morphology returns ('ind', 'reg') if dir==1 else ('reg', 'ind')
        # Here we only care WHETHER kata/form are Indonesian — they are iff dir==1.
        indonesian_columns = ['kata', 'form'] if is_from_indonesia == 1 else []

        out_df    = df.copy()
        audit_df  = df.copy()
        audit_df['direction'] = is_from_indonesia

        if not indonesian_columns:
            # Regional→Indonesian dict: passthrough.
            out_df.to_csv(output_file, index=False)
            audit_df['kata_corrected'] = False
            audit_df['form_corrected'] = False
            audit_df.to_csv(audit_file, index=False)
            return f'Passthrough (dir=0): {file_name} ({len(out_df)} rows)'

        n_changed_total = 0
        for col in indonesian_columns:
            original = df[col].astype(str).where(df[col].notna(), '')
            corrected = original.apply(fix_cell)
            out_df[col] = corrected
            audit_df[f'{col}_original']  = original
            audit_df[col]                = corrected
            audit_df[f'{col}_corrected'] = original != corrected
            n_changed_total += int((original != corrected).sum())

        out_df.to_csv(output_file, index=False)
        audit_df.to_csv(audit_file, index=False)
        return f'Processed: {file_name} ({len(out_df)} rows, {n_changed_total} corrections in {indonesian_columns})'

    except Exception as e:
        return f'Error in {file_name}: {e}'


morph_files = [
    f for f in os.listdir(MORPH_SRC)
    if f.endswith('.csv') and not f.endswith('_audit.csv')
    and not f.startswith('_')
]
print(f'Found {len(morph_files)} Morphology files to process')

morph_results = []
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(process_morph, f) for f in morph_files]
    for count, future in enumerate(concurrent.futures.as_completed(futures), 1):
        msg = future.result()
        morph_results.append(msg)
        print(f'[{count}/{len(morph_files)}] {msg}')

Found 165 Morphology files to process
[1/165] Skip: 0_DerivedBillex.csv missing kata/form columns
[2/165] Skip: 12_DerivedBillex.csv missing kata/form columns
[3/165] Skip: 10_DerivedBillex.csv missing kata/form columns
[4/165] Skip: 11_DerivedBillex.csv missing kata/form columns
[5/165] Skip: 14_DerivedBillex.csv missing kata/form columns
[6/165] Skip: 13_DerivedBillex.csv missing kata/form columns
[7/165] Passthrough (dir=0): 11_Morphology.csv (1805 rows)
[8/165] Skip: 16_DerivedBillex.csv missing kata/form columns
[9/165] Skip: 15_DerivedBillex.csv missing kata/form columns
[10/165] Skip: 17_DerivedBillex.csv missing kata/form columns
[11/165] Passthrough (dir=0): 17_Morphology.csv (237 rows)
[12/165] Skip: 18_DerivedBillex.csv missing kata/form columns
[13/165] Passthrough (dir=0): 18_Morphology.csv (3098 rows)
[14/165] Skip: 19_DerivedBillex.csv missing kata/form columns
[15/165] Processed: 0_Morphology.csv (16 rows, 18 corrections in ['kata', 'form'])
[16/165] Skip: 1_DerivedBill

## 7. Per-dictionary correction summaries

Re-reads the audit files to build a wide summary table. Useful for the thesis
appendix to report, per dictionary, how many cells the spellchecker changed in
each resource. Saves as `_billex_summary.csv` and `_morph_summary.csv` next to
the output files.

In [34]:
def billex_summary_row(audit_path: Path) -> Optional[dict]:
    dict_id = parse_dict_id(audit_path.name)
    if dict_id is None:
        return None
    df = pd.read_csv(audit_path)
    if df.empty or 'direction' not in df.columns:
        return {
            'dict_id': dict_id,
            'direction': -1,
            'indonesian_column': '',
            'rows': len(df),
            'corrected_cells': 0,
        }
    direction = int(df['direction'].iloc[0])
    ind_col = 'kata_asal' if direction == 1 else 'kata_tujuan'
    flag = f'{ind_col}_corrected'
    return {
        'dict_id': dict_id,
        'direction': direction,
        'indonesian_column': ind_col,
        'rows': len(df),
        'corrected_cells': int(df[flag].sum()) if flag in df.columns else 0,
    }


def morph_summary_row(audit_path: Path) -> Optional[dict]:
    dict_id = parse_dict_id(audit_path.name)
    if dict_id is None:
        return None
    df = pd.read_csv(audit_path)
    if df.empty or 'direction' not in df.columns:
        return {
            'dict_id': dict_id,
            'direction': -1,
            'rows': len(df),
            'kata_corrected_cells': 0,
            'form_corrected_cells': 0,
            'total_corrected_cells': 0,
        }
    direction = int(df['direction'].iloc[0])
    kata_changed = int(df['kata_corrected'].sum()) if 'kata_corrected' in df.columns else 0
    form_changed = int(df['form_corrected'].sum()) if 'form_corrected' in df.columns else 0
    return {
        'dict_id': dict_id,
        'direction': direction,
        'rows': len(df),
        'kata_corrected_cells': kata_changed,
        'form_corrected_cells': form_changed,
        'total_corrected_cells': kata_changed + form_changed,
    }

## 8. Cross-resource sanity checks

Two consistency checks after the run:

1. Every Billex/Morph file in the source directory should now have a matching
   output file.
2. The number of rows must be **exactly** preserved per file. The
   spellchecker only edits cell values; it must not add or drop rows.

In [35]:
def check_row_parity(src_dir: Path, dst_dir: Path, suffix: str) -> pd.DataFrame:
    records = []
    for src in sorted(src_dir.glob(f'*_{suffix}.csv')):
        if src.name.endswith('_audit.csv'):
            continue
        dst = dst_dir / src.name
        if not dst.exists():
            records.append({'file': src.name, 'src_rows': -1, 'dst_rows': -1, 'status': 'MISSING'})
            continue
        try:
            n_src = len(pd.read_csv(src))
            n_dst = len(pd.read_csv(dst))
        except Exception as e:
            records.append({'file': src.name, 'src_rows': -1, 'dst_rows': -1, 'status': f'ERR: {e}'})
            continue
        status = 'OK' if n_src == n_dst else 'MISMATCH'
        records.append({'file': src.name, 'src_rows': n_src, 'dst_rows': n_dst, 'status': status})
    return pd.DataFrame(records)


billex_parity = check_row_parity(BILLEX_SRC, BILLEX_DST, 'Billex')
morph_parity  = check_row_parity(MORPH_SRC,  MORPH_DST,  'Morphology')

print('Billex parity:')
print(f"  OK:       {(billex_parity['status'] == 'OK').sum()}")
print(f"  MISSING:  {(billex_parity['status'] == 'MISSING').sum()}")
print(f"  MISMATCH: {(billex_parity['status'] == 'MISMATCH').sum()}")
bad = billex_parity[billex_parity['status'] != 'OK']
if not bad.empty:
    display(bad)

print('\nMorphology parity:')
print(f"  OK:       {(morph_parity['status'] == 'OK').sum()}")
print(f"  MISSING:  {(morph_parity['status'] == 'MISSING').sum()}")
print(f"  MISMATCH: {(morph_parity['status'] == 'MISMATCH').sum()}")
bad = morph_parity[morph_parity['status'] != 'OK']
if not bad.empty:
    display(bad)

Billex parity:
  OK:       81
  MISSING:  1
  MISMATCH: 0


,file,src_rows,dst_rows,status
39,47_Billex.csv,-1,-1,MISSING



Morphology parity:
  OK:       81
  MISSING:  1
  MISMATCH: 0


,file,src_rows,dst_rows,status
39,47_Morphology.csv,-1,-1,MISSING


## 9. Spot-check a few corrections

Prints sample original to corrected pairs from one Indonesian-source Billex
(`dict 4`, Jambi) and one Indonesian-source Morphology (`dict 18`, Javanese).
Sanity check that the corrections look reasonable before downstream stages
consume the output.

In [36]:
def spot_check_billex(dict_id: str, n: int = 15) -> None:
    audit = BILLEX_DST / f'{dict_id}_Billex_audit.csv'
    if not audit.exists():
        print(f'  (no audit file for dict {dict_id})')
        return
    df = pd.read_csv(audit)
    direction = int(df['direction'].iloc[0]) if 'direction' in df.columns else 1
    ind_col = 'kata_asal' if direction == 1 else 'kata_tujuan'
    flag = f'{ind_col}_corrected'
    orig = f'{ind_col}_original'
    if flag not in df.columns:
        print(f'  (no correction flag in dict {dict_id})')
        return
    changed = df[df[flag]][[orig, ind_col]].drop_duplicates().head(n)
    print(f'  Dict {dict_id} Billex ({ind_col}, dir={direction}): {df[flag].sum()} cells changed')
    print(changed.to_string(index=False))


def spot_check_morph(dict_id: str, n: int = 15) -> None:
    audit = MORPH_DST / f'{dict_id}_Morphology_audit.csv'
    if not audit.exists():
        print(f'  (no audit file for dict {dict_id})')
        return
    df = pd.read_csv(audit)
    direction = int(df['direction'].iloc[0]) if 'direction' in df.columns else 1
    if direction != 1:
        print(f'  Dict {dict_id} Morphology: dir={direction}, passthrough (no Indonesian columns)')
        return
    changed_rows = df[df.get('kata_corrected', False) | df.get('form_corrected', False)]
    cols = ['kata_original', 'kata', 'form_original', 'form']
    cols = [c for c in cols if c in df.columns]
    print(f'  Dict {dict_id} Morphology (dir=1): '
          f"kata={df.get('kata_corrected', pd.Series(dtype=bool)).sum()} "
          f"form={df.get('form_corrected', pd.Series(dtype=bool)).sum()} corrections")
    print(changed_rows[cols].drop_duplicates().head(n).to_string(index=False))


print('=== Billex sample (dict 4) ===')
spot_check_billex('4')
print('\n=== Morphology sample (dict 18) ===')
spot_check_morph('18')

=== Billex sample (dict 4) ===
  Dict 4 Billex (kata_asal, dir=1): 1425 cells changed
kata_asal_original   kata_asal
       mengabdikan mengabaikan
         melalekan   melakukan
        manyimpang  menyimpang
             kiabu        kabu
          sukonian    sokongan
            acagha       acara
        pengacagha   pengacara
          ndulikan   pedulikan
        mengadokan  mengadakan
            adodak      adonan
          beghadap    berhadap
           ngadili   mengadili
           campugh      campur
         campughan    campuran
          ngadukan    ngajukan

=== Morphology sample (dict 18) ===
  Dict 18 Morphology: dir=0, passthrough (no Indonesian columns)


## 10. Notes

- **Why a separate `13.` / `14.` directory?** Mirrors the convention used for
  Parcor (`12. Parallel Corpus - Spelling Checker`) so each resource keeps
  pre- and post-spellchecker copies side by side. Downstream notebooks can
  switch between the two by changing one path.

- **Switching dictionary strategy.** The same JSON dictionary toggle works
  here — flip `DICT_PATH` to `indonesian_dict_strategy_B.json` or
  `indonesian_dict_strategy_C.json` and rerun. Recommended to record which
  strategy was used by saving the active path into `_billex_summary.csv` /
  `_morph_summary.csv` as a column for the thesis appendix (left as a TODO).

- **What this notebook deliberately does NOT do.**
  - It does not re-run any of the upstream batch fixes (`Fix Billex Headword
    Batch`, `Fix Morphology Batch`). Those must run first to produce the
    `*_Fixed` directories this notebook reads from.
  - It does not touch Parcor.
  - It does not spellcheck regional-language content under any circumstances,
    even when the Indonesian dictionary contains plausible-looking matches.
